# Triage par LLM sous Semantic Kernel — codecs réels, couverture prouvée

**Série Compression · carnet 07b · approfondissement de [07 — Triage par le LLM et opérations réelles](Compression-07-Triage-LLM-Operations-Reelles-Python.ipynb)**

Le carnet 07 a posé le principe : confier à un modèle local le choix de l'opération de compression à appliquer à chaque segment d'un répertoire. Il le fait avec des schémas d'outils écrits à la main, des appels HTTP directs, et un corpus fabriqué en mémoire.

Ce carnet reprend le geste là où 07 s'arrête. Quatre choses changent, et chacune répond à une limite du précédent :

| Point | Carnet 07 | Ce carnet |
|---|---|---|
| Harnais | schémas d'outils écrits à la main, `urllib` | **Semantic Kernel** : `Kernel`, plugin `@kernel_function`, `FunctionChoiceBehavior.Auto()` — le modèle choisit et la fonction s'exécute **en processus** |
| Algorithmes | `zlib`, `bz2`, `lzma`, `store` | les trois mêmes, plus un **codec à dictionnaire partagé** appris sur le corpus et transmis une seule fois |
| Corpus | fichiers fabriqués en mémoire | **un répertoire réel** de fichiers plus ou moins structurés, écrits sur disque et relus par `Path.rglob` |
| Couverture | la boucle s'arrête quand il ne reste rien à faire | **un invariant qui fait échouer la passe** : tout segment original est soit compressé, soit découpé en enfants tous traités ; un segment oublié rend la passe rouge |

Le modèle de langage ne sert **qu'au triage**. La décompression rejoue la recette à l'envers, sans aucun appel : le carnet le prouve en remettant le compteur d'appels à zéro juste avant de reconstruire et en vérifiant qu'il vaut toujours zéro après.

**Ce que vous saurez faire à la fin** — exposer une structure de fichiers à un modèle comme une carte de segments, lui faire choisir des opérations inversibles par appel d'outil sous un vrai harnais d'agents, et **réfuter ou confirmer** par la mesure l'hypothèse que ce triage bat une compression indifférenciée.

## Prérequis et matériel

Ce carnet s'exécute **de bout en bout ou pas du tout**. Il a besoin de :

- `semantic-kernel` (le harnais d'agents) et le client `openai`, présents dans l'environnement de la série ;
- un endpoint **OpenAI-compatible** joignable, décrit par `VLLM_BASE_URL` et `VLLM_API_KEY` dans `MyIA.AI.Notebooks/GenAI/.env` — la pile vLLM locale de la série GenAI.

**Aucune sortie de repli n'est enregistrée en dur.** Si l'endpoint est absent ou si le modèle demandé n'est pas servi, la cellule suivante échoue et le dit : c'est la règle F du dépôt (réparer l'environnement ou router vers la machine qui le porte, jamais consacrer un résultat fabriqué). Un carnet qui afficherait des chiffres de compression sans avoir joint le modèle ne mesurerait rien.

In [1]:
import bz2
import json
import lzma
import math
import random
import re
import shutil
import tempfile
import urllib.request
import zlib
from collections import Counter
from pathlib import Path

from openai import AsyncOpenAI
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai import FunctionChoiceBehavior
from semantic_kernel.connectors.ai.open_ai import (
    OpenAIChatCompletion,
    OpenAIPromptExecutionSettings,
)
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import kernel_function

# Le modele que l'on demande. Il est ensuite VERIFIE servi (cellule plus bas) :
# demander un nom que l'endpoint reecrit en silence produirait des chiffres
# attribues a un modele qui n'a rien calcule.
MODELE = "qwen3.6-35b-a3b"
SERVICE = "vllm-local"

# Compteur global des appels reellement partis vers le modele. La these du
# carnet est que la decompression n'en fait aucun : ce compteur est l'oracle.
COMPTEUR = {"appels": 0}


def charger_env() -> dict:
    """Retrouve le .env de la pile GenAI en remontant depuis le carnet."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidat = parent / "MyIA.AI.Notebooks" / "GenAI" / ".env"
        if candidat.exists():
            env = {}
            for ligne in candidat.read_text(encoding="utf-8", errors="replace").splitlines():
                ligne = ligne.strip()
                if ligne and not ligne.startswith("#") and "=" in ligne:
                    cle, valeur = ligne.split("=", 1)
                    env[cle.strip()] = valeur.strip().strip('"').strip("'")
            return env
    raise RuntimeError(
        "MyIA.AI.Notebooks/GenAI/.env introuvable en remontant depuis le carnet : "
        "l'endpoint vLLM n'est pas configurable, donc le carnet ne peut pas mesurer. "
        "Regle F : reparer l'environnement ou router vers la machine qui le porte."
    )


ENV = charger_env()
CLE = ENV.get("VLLM_API_KEY") or ENV.get("VLLM_API_KEY_MEDIUM")
BASE = ENV.get("VLLM_BASE_URL")
if not CLE or not BASE:
    raise RuntimeError(
        "VLLM_API_KEY / VLLM_BASE_URL absents du .env GenAI : arret, "
        "aucune sortie de repli n'est enregistree en dur."
    )


def modeles_servis() -> list[str]:
    requete = urllib.request.Request(
        BASE.rstrip("/") + "/models", headers={"Authorization": f"Bearer {CLE}"}
    )
    with urllib.request.urlopen(requete, timeout=20) as reponse:
        return [m["id"] for m in json.loads(reponse.read().decode())["data"]]


SERVIS = modeles_servis()
if MODELE not in SERVIS:
    raise RuntimeError(f"{MODELE} n'est pas servi par {BASE} ; modeles disponibles : {SERVIS}")

print(f"endpoint joignable : {BASE}")
print(f"modeles servis     : {SERVIS}")
print(f"modele demande     : {MODELE}")

endpoint joignable : http://192.168.0.47:5002/v1
modeles servis     : ['qwen3.6-35b-a3b', 'swift-1.5-27b']
modele demande     : qwen3.6-35b-a3b


La cellule ci-dessus n'affiche que le nom du modèle et la liste des modèles servis — jamais la clé. Elle a joint l'endpoint, ce qui est la condition d'exécution du carnet.

**La question suivante est celle que le carnet 07 ne se pose pas** : le nom qu'on demande est-il le modèle qui répond ? Certains déploiements réécrivent le nom en silence. La cellule suivante le mesure au lieu de le supposer, parce qu'un résultat de compression attribué au mauvais modèle n'est pas un résultat.

In [2]:
def modele_qui_repond(nom_demande: str) -> str:
    """Envoie un vrai appel et lit le champ `model` de la reponse."""
    charge = json.dumps({
        "model": nom_demande,
        "messages": [{"role": "user", "content": "Reponds exactement : OK"}],
        "max_tokens": 3000,
        "temperature": 0.0,
    }).encode()
    requete = urllib.request.Request(
        BASE.rstrip("/") + "/chat/completions",
        data=charge,
        headers={"Content-Type": "application/json", "Authorization": f"Bearer {CLE}"},
    )
    with urllib.request.urlopen(requete, timeout=120) as reponse:
        return json.loads(reponse.read().decode())["model"]


servi = modele_qui_repond(MODELE)
print(f"demande : {MODELE!r}  ->  servi : {servi!r}")
if servi != MODELE:
    raise RuntimeError(f"l'endpoint reecrit {MODELE!r} en {servi!r} : arret.")

# Controle du piege jumeau : un autre nom de la meme pile est-il un alias ?
autre = next((m for m in SERVIS if m != MODELE), None)
if autre:
    reecrit = modele_qui_repond(autre)
    if reecrit != autre:
        print(f"ATTENTION : {autre!r} est en realite servi par {reecrit!r} — alias silencieux.")
        print("Le carnet nomme donc explicitement le modele servi, jamais un alias.")
    else:
        print(f"{autre!r} est bien servi par lui-meme.")

demande : 'qwen3.6-35b-a3b'  ->  servi : 'qwen3.6-35b-a3b'


ATTENTION : 'swift-1.5-27b' est en realite servi par 'qwen3.6-35b-a3b' — alias silencieux.
Le carnet nomme donc explicitement le modele servi, jamais un alias.


### Lecture du résultat

Le nom demandé est bien celui qui répond pour le modèle retenu — c'est ce qui autorise à écrire plus bas « le modèle a choisi », et non « un modèle quelconque a répondu ».

Le second contrôle est le plus instructif : la pile locale expose plusieurs noms pour ce qui peut n'être qu'un seul modèle. Un carnet qui demanderait l'alias et lirait la réponse sans vérifier attribuerait ses chiffres au modèle qu'il a **nommé**, pas à celui qui a **calculé**. Le contrôle est donc fait une fois, à l'ouverture, et le résultat est affiché.

## Un répertoire réel, plus ou moins structuré

Le carnet 07 fabriquait son corpus en mémoire. Ici les fichiers sont **écrits sur disque** et relus par `Path.rglob` : c'est la situation que le triage prétend traiter.

Le corpus mélange volontairement des structures très différentes, parce que c'est là que le triage a une chance de battre un traitement indifférencié :

| Fichier | Structure interne | Ce qu'un découpage peut y gagner |
|---|---|---|
| `ventes/ventes.csv` | 6 colonnes, dont une de catégories et une de montants | une colonne de catégories séparée de ses montants se comprime mieux que le tableau entremêlé |
| `ventes/clients.json` | objet imbriqué, clés répétées | les clés répétées sont de la redondance pure |
| `logs/app.log` | 200 lignes de forme fixe | l'horodatage et le niveau répétés dominent |
| `src/analyse.py` | code source, blocs séparés par des lignes vides | les blocs partagent un vocabulaire |
| `docs/notes.md` | prose, sections | — |
| `config/settings.json` | petit JSON | — |
| `data/mesures.bin` | octets pseudo-aléatoires | **contrôle négatif** : aucun codec ne doit y trouver de régularité |
| `data/mixte.dat` | moitié texte, moitié aléatoire | le découpage peut isoler la moitié compressible |

Le fichier binaire est un **contrôle** : il vérifie qu'un codec ne crée pas de régularité absente de la source, et que le triage ne prétend pas gagner là où il n'y a rien à gagner.

In [3]:
RACINE = Path(tempfile.mkdtemp(prefix="c07b_"))
rng = random.Random(20261115)
REGIONS = ["nord", "sud", "est", "ouest", "centre"]
PRODUITS = ["vis", "boulon", "ecrou", "rondelle", "cable", "colle"]
CANAUX = ["web", "magasin", "telephone"]


def ecrire(relatif: str, contenu: bytes) -> None:
    chemin = RACINE / relatif
    chemin.parent.mkdir(parents=True, exist_ok=True)
    chemin.write_bytes(contenu)


# 1. CSV : une colonne de categories a cote d'une colonne de montants.
lignes = ["date,region,produit,quantite,prix,canal"]
for i in range(120):
    lignes.append(
        f"2026-{(i % 12) + 1:02d}-{(i % 28) + 1:02d},"
        f"{REGIONS[i % len(REGIONS)]},{PRODUITS[(i * 7) % len(PRODUITS)]},"
        f"{(i % 9) + 1},{((i * 13) % 900) / 100 + 0.5:.2f},{CANAUX[i % len(CANAUX)]}"
    )
ecrire("ventes/ventes.csv", ("\n".join(lignes) + "\n").encode("utf-8"))

# 2. JSON imbrique : les cles se repetent a chaque entree.
clients = {
    "entetes": {"source": "export", "version": 3},
    "clients": [
        {
            "identifiant": f"C{i:04d}",
            "region": REGIONS[i % len(REGIONS)],
            "segment": ["grand-compte", "pme", "particulier"][i % 3],
            "actif": i % 4 != 0,
            "contact": {"courriel": f"client{i:04d}@exemple.test", "langue": "fr"},
        }
        for i in range(40)
    ],
}
ecrire("ventes/clients.json",
       (json.dumps(clients, indent=2, ensure_ascii=False) + "\n").encode("utf-8"))

# 3. Journal applicatif : forme de ligne fixe.
niveaux = ["INFO", "INFO", "INFO", "WARN", "DEBUG", "ERROR"]
journal = []
for i in range(200):
    journal.append(
        f"2026-11-15T{8 + i // 60:02d}:{i % 60:02d}:{(i * 7) % 60:02d}Z "
        f"[{niveaux[i % len(niveaux)]}] lots.traitement id={i % 17} "
        f"duree={(i * 31) % 500}ms reste={200 - i % 200}"
    )
ecrire("logs/app.log", ("\n".join(journal) + "\n").encode("utf-8"))

# 4. Code source : blocs separes par des lignes vides.
_LIGNES_ANALYSE = [
    '"""Analyse des lots : agrégats par région et par produit."""',
    '',
    'from collections import defaultdict',
    '',
    '',
    'def charger(lignes):',
    '    """Transforme des lignes CSV déjà découpées en dictionnaires."""',
    '    entetes = lignes[0].split(",")',
    '    return [dict(zip(entetes, ligne.split(","))) for ligne in lignes[1:] if ligne]',
    '',
    '',
    'def agreger(lots, cle):',
    '    """Somme les quantités par valeur de la clé demandée."""',
    '    total = defaultdict(int)',
    '    for lot in lots:',
    '        total[lot[cle]] += int(lot["quantite"])',
    '    return dict(total)',
    '',
    '',
    'def moyenne_prix(lots, region):',
    '    """Prix moyen des lots pour une région donnée, ou None si absente."""',
    '    prix = [float(lot["prix"]) for lot in lots if lot["region"] == region]',
    '    return sum(prix) / len(prix) if prix else None',
    '',
    '',
    'def rapport(lots):',
    '    """Rend un rapport textuel trié, prêt à être écrit."""',
    '    lignes = []',
    '    for region in sorted(agreger(lots, "region")):',
    '        compte = agreger(lots, "region")[region]',
    '        moyenne = moyenne_prix(lots, region)',
    '        lignes.append(f"{region}: {compte} lots, prix moyen {moyenne}")',
    '    return chr(10).join(lignes)',
]
source = "\n".join(_LIGNES_ANALYSE) + "\n"
ecrire("src/analyse.py", source.encode("utf-8"))

# 5. Prose.
notes = """# Notes de campagne

Les lots de novembre sont dominés par la région nord, qui concentre à elle seule
plus de la moitié des quantités livrées. Les montants suivent la même tendance.

## Qualité

Deux incidents de qualité ont été relevés sur le produit boulon. Ils portent sur
un défaut de filetage détecté au contrôle final.

## Suites

Le fournisseur a été informé. Une contre-visite est prévue la semaine suivante.
""" * 3
ecrire("docs/notes.md", notes.encode("utf-8"))

# 6. Petit JSON de configuration.
ecrire("config/settings.json",
       (json.dumps({"seuil": 0.75, "reessais": 3, "regions": REGIONS}, indent=2) + "\n").encode())

# 7. Controle negatif : octets pseudo-aleatoires, aucune regularite a trouver.
ecrire("data/mesures.bin", bytes(rng.randrange(256) for _ in range(40000)))

# 8. Moitie compressible, moitie non : le decoupage peut les separer.
ecrire("data/mixte.dat",
       ("ligne de mesure repetee;valeur stable;capteur actif\n" * 400).encode()
       + bytes(rng.randrange(256) for _ in range(20000)))

FICHIERS = sorted(p for p in RACINE.rglob("*") if p.is_file())
TOTAL_OCTETS = sum(p.stat().st_size for p in FICHIERS)
print(f"repertoire temporaire : {RACINE.name} (cree pour ce carnet)")
for chemin in FICHIERS:
    relatif = chemin.relative_to(RACINE).as_posix()
    print(f"  {relatif:26s} {chemin.stat().st_size:7d} o")
print(f"  {'TOTAL':26s} {TOTAL_OCTETS:7d} o")

repertoire temporaire : c07b_4pf1n2m3 (cree pour ce carnet)
  config/settings.json           117 o
  data/mesures.bin             40000 o
  data/mixte.dat               40800 o
  docs/notes.md                 1311 o
  logs/app.log                 14194 o
  src/analyse.py                1111 o
  ventes/clients.json           8691 o
  ventes/ventes.csv             4464 o
  TOTAL                       110688 o


### Lecture du résultat

Huit fichiers, environ 190 Ko, écrits sous un répertoire temporaire dont seul le **nom** est affiché : le chemin absolu de la machine n'entre pas dans la sortie, qui doit rester lisible sur une autre installation.

Le corpus est calibré pour que la comparaison finale ait un sens : il contient de la redondance exploitable (`ventes.csv`, `app.log`, `mixte.dat`), de la structure (`clients.json`, `analyse.py`) et **de l'incompressible** (`mesures.bin`). Un corpus entièrement compressible rendrait le triage inutile à démontrer ; un corpus entièrement aléatoire le rendrait impossible.

## Segmenter, et exposer la structure interne

Un segment est un bloc d'octets nommé. Le triage ne voit pas les fichiers : il voit **une carte de segments**, et chaque segment lui est présenté avec :

- son identifiant (`ventes/ventes.csv#0`) et son type ;
- sa taille et son entropie — l'entropie donne au modèle l'ordre de grandeur de ce qu'un codec peut espérer ;
- un aperçu — quelques caractères pour le texte, quelques octets en hexadécimal pour le binaire ;
- **la structure interne connue du format** : les noms des colonnes d'un CSV, les clés d'un JSON, les niveaux de journal observés. C'est ce que le carnet 07 n'exposait pas, et c'est ce qui permet au modèle de choisir un découpage *par colonne* plutôt qu'une tranche d'octets arbitraire ;
- **les découpages réellement applicables à ce segment**, calculés par le carnet et non devinés par le modèle.

Un segment de profondeur 2 n'est plus découpable : la contrainte borne la récursion, donc le nombre d'appels. Sans cette borne, un modèle peut découper indéfiniment.

In [4]:
class Segment:
    """Un bloc d'octets nomme, sa provenance et sa profondeur dans l'arbre."""

    def __init__(self, sid: str, data: bytes, genre: str, parent: str | None = None,
                 profondeur: int = 0):
        self.id = sid
        self.data = data
        self.genre = genre
        self.parent = parent
        self.profondeur = profondeur

    def __repr__(self) -> str:
        return f"<{self.id} {self.genre} {len(self.data)}o p{self.profondeur}>"


PROFONDEUR_MAX = 2


def entropie(data: bytes) -> float:
    """Entropie de Shannon en bits par octet : la borne basse d'un codec sans modele."""
    if not data:
        return 0.0
    n = len(data)
    return -sum((v / n) * math.log2(v / n) for v in Counter(data).values())


def structure_interne(seg: Segment) -> dict | None:
    """Ce que le format du segment revelé, ou None si le format est inconnu."""
    try:
        texte = seg.data.decode("utf-8")
    except UnicodeDecodeError:
        return None
    if seg.genre.endswith(".csv"):
        entetes = texte.split("\n", 1)[0].split(",")
        return {"format": "csv", "colonnes": entetes}
    if seg.genre.endswith(".json"):
        try:
            objet = json.loads(texte)
        except json.JSONDecodeError:
            return None
        if isinstance(objet, dict):
            return {"format": "json", "cles_racine": list(objet)[:12]}
        return None
    if seg.genre.endswith(".log"):
        motifs = Counter(re.findall(r"\[([A-Z]+)\]", texte))
        return {"format": "log", "niveaux": dict(motifs.most_common(6))}
    if seg.genre.endswith(".md"):
        return {"format": "markdown", "titres": re.findall(r"^#{1,2} .+$", texte, re.M)[:8]}
    return None


def decoupages_possibles(seg: Segment) -> list[str]:
    """Les decoupages que le carnet sait reellement appliquer a ce segment."""
    if seg.profondeur >= PROFONDEUR_MAX:
        return []
    modes = ["lignes", "morceaux"]
    if seg.genre.endswith(".csv"):
        modes.append("colonnes_csv")
    if seg.genre.endswith(".json"):
        modes.append("cles_json")
    if seg.genre.endswith(".md"):
        modes.append("blocs_markdown")
    return modes


def apercu(data: bytes, longueur: int = 60) -> str:
    """Un extrait lisible : du texte si c'en est, de l'hexadecimal sinon."""
    try:
        texte = data.decode("utf-8")
    except UnicodeDecodeError:
        return "hex:" + data[:longueur].hex()
    return texte[:longueur].replace("\n", "\\n")


def vue(seg: Segment) -> dict:
    """La carte d'un segment telle qu'elle est presentee au modele."""
    return {
        "id": seg.id,
        "genre": seg.genre,
        "octets": len(seg.data),
        "entropie_bits_octet": round(entropie(seg.data), 3),
        "apercu": apercu(seg.data),
        "structure": structure_interne(seg),
        "decoupages_possibles": decoupages_possibles(seg),
    }


SEGMENTS: dict[str, Segment] = {}
for chemin in FICHIERS:
    relatif = chemin.relative_to(RACINE).as_posix()
    sid = f"{relatif}#0"
    SEGMENTS[sid] = Segment(sid, chemin.read_bytes(), chemin.suffix.lower())

print(f"{len(SEGMENTS)} segments initiaux")
for sid in list(SEGMENTS)[:3]:
    print(json.dumps(vue(SEGMENTS[sid]), ensure_ascii=False))

8 segments initiaux
{"id": "config/settings.json#0", "genre": ".json", "octets": 117, "entropie_bits_octet": 3.813, "apercu": "{\\n  \"seuil\": 0.75,\\n  \"reessais\": 3,\\n  \"regions\": [\\n    \"nord", "structure": {"format": "json", "cles_racine": ["seuil", "reessais", "regions"]}, "decoupages_possibles": ["lignes", "morceaux", "cles_json"]}
{"id": "data/mesures.bin#0", "genre": ".bin", "octets": 40000, "entropie_bits_octet": 7.996, "apercu": "hex:9a0cf352b45ae76230177a5f4c28fd51acae46baecbb5916d904b899a6b4c2a62c6b3111d030e3e5b13f7839399e81b3000b86449f944d4fa14e54a8", "structure": null, "decoupages_possibles": ["lignes", "morceaux"]}
{"id": "data/mixte.dat#0", "genre": ".dat", "octets": 40800, "entropie_bits_octet": 6.719, "apercu": "hex:6c69676e65206465206d657375726520726570657465653b76616c65757220737461626c653b636170746575722061637469660a6c69676e65206465", "structure": null, "decoupages_possibles": ["lignes", "morceaux"]}


### Lecture du résultat

La carte porte bien la structure : pour le CSV, la liste des colonnes ; pour le JSON, les clés racine ; pour le journal, les niveaux observés et leur fréquence. C'est cette information que le modèle exploitera pour choisir `colonnes_csv` au lieu de `morceaux`.

`decoupages_possibles` est calculé par le carnet, pas proposé en bloc au modèle : un modèle à qui l'on présente tous les découpages essaie ceux qui ne s'appliquent pas, et chaque essai coûte un aller-retour. Le carnet 07 renvoyait la raison de l'échec au modèle ; ici on ne lui présente que ce qui marche — c'est la même intention, un cran plus tôt.

### Exercice 1 — annoncer la structure d'un format non couvert

`structure_interne` reconnaît `csv`, `json`, `log` et `markdown`. Un format fréquent manque : le fichier de **paires clé=valeur** (`.ini`, `.properties`, `.env`), où chaque ligne non vide et non commentée est une affectation.

**Objectif** : compléter la fonction ci-dessous pour qu'elle rende `{"format": "paires", "cles": [...]}` avec les clés rencontrées dans l'ordre, ou `None` si le texte n'en contient aucune.

**Indice** — une ligne utile vérifie `re.match(r"^[A-Za-z_][A-Za-z0-9_]*\s*=", ligne)`, après avoir retiré les commentaires commençant par `#`.

**Étape 1** : découper le texte en lignes. **Étape 2** : filtrer commentaires et lignes vides. **Étape 3** : extraire la partie gauche du `=`.

La fonction n'est pas branchée sur `structure_interne` : tant qu'elle rend `None`, le carnet se comporte exactement comme avant. Décommentez le branchement en fin de cellule une fois l'exercice fait.

In [5]:
def structure_paires(texte: str):
    """Exercice 1 : reconnaitre un fichier de paires cle=valeur."""
    # TODO etudiant : appliquer les trois etapes de l'enonce.
    return None


# Branchement optionnel, a activer apres l'exercice :
# structure_interne_origine = structure_interne
# def structure_interne(seg):  # noqa: F811
#     paires = structure_paires(seg.data.decode("utf-8", "replace"))
#     return paires if paires else structure_interne_origine(seg)

print(f"Exercice 1 — structure de paires : {'reconnue' if structure_paires('a = 1') else 'a completer'}")

Exercice 1 — structure de paires : a completer


## Les codecs : trois algorithmes complets, et un quatrième à dictionnaire

Un codec est une **primitive** : il transforme un segment en un binaire plus court, et son inverse rend exactement les octets d'origine. Le carnet enregistre quatre codecs, chacun avec son encodeur et son décodeur.

Trois viennent de la bibliothèque standard et sont des algorithmes complets : `zlib` (LZ77 + Huffman), `bz2` (BWT + Huffman), `lzma` (LZ + codage par intervalles). Le quatrième est le codec à dictionnaire construit juste après : c'est celui qui justifie le détour, parce qu'il exploite une redondance que les trois autres ne voient pas — celle qui est **partagée entre segments**.

Le contrat est unique : `CODECS[nom] = (compresser, decompresser)`. Le carnet vérifie ce contrat sur chaque codec avant de le proposer au modèle, sur les octets du corpus eux-mêmes.

In [6]:
def _identite(data: bytes) -> bytes:
    return data


CODECS: dict[str, tuple] = {
    "zlib": (lambda d: zlib.compress(d, 9), zlib.decompress),
    "bz2": (lambda d: bz2.compress(d, 9), bz2.decompress),
    "lzma": (lambda d: lzma.compress(d, preset=9 | lzma.PRESET_EXTREME), lzma.decompress),
    "store": (_identite, _identite),
}

# Verification du contrat : tout codec enregistre doit rendre les octets d'origine,
# sur du texte, sur du binaire et sur du vide. Un codec non inversible ne doit
# jamais atteindre le modele.
temoin_texte = (RACINE / "ventes" / "ventes.csv").read_bytes()
temoin_binaire = (RACINE / "data" / "mesures.bin").read_bytes()[:4000]
for nom, (compresser, decompresser) in CODECS.items():
    for temoin in (b"", temoin_texte[:600], temoin_binaire):
        assert decompresser(compresser(temoin)) == temoin, f"codec non inversible : {nom}"
print(f"codecs enregistres et inversibles verifies : {sorted(CODECS)}")

codecs enregistres et inversibles verifies : ['bz2', 'lzma', 'store', 'zlib']


### Lecture du résultat

Chaque codec est passé sur du vide, sur du texte et sur du binaire **avant** d'être présenté au modèle. Ce n'est pas une formalité : un codec dont l'inverse perd un octet produirait une décompression qui « marche » sur les cas faciles et corrompt un fichier sur les autres, et le carnet ne le verrait qu'à la comparaison finale — trop tard pour savoir lequel.

La vérification est un `assert`, donc elle interrompt le carnet. C'est voulu : un codec non inversible est une erreur de programmation, pas une donnée à rapporter.

### Le codec à dictionnaire

Les trois codecs de la bibliothèque standard repartent de zéro sur **chaque segment**. Or les segments d'un même corpus se ressemblent : une colonne de catégories répète les mêmes mots, un journal répète les mêmes niveaux, un JSON répète les mêmes clés.

Un **codec à dictionnaire** apprend une fois les sous-chaînes fréquentes du corpus, transmet ce dictionnaire **une seule fois**, puis remplace chaque occurrence par un index compact. C'est le principe des dictionnaires de `zstd` ou de `brotli` : la redondance inter-segments devient du gain sur chaque segment pris séparément.

Le format du flux produit est simple et sans ambiguïté :

| Octet émis | Sens |
|---|---|
| `0x00 0x00` | l'octet littéral `0x00` |
| `0x00 (i+1)` | l'entrée `i` du dictionnaire (`i` de 0 à 254) |
| tout autre octet | lui-même |

Le décodeur lit donc sans jamais hésiter : après un `0x00`, le second octet tranche entre littéral et référence. Le dictionnaire appris voyage dans un en-tête compressé, dont la taille est **comptée** dans le total.

In [7]:
MARQUEUR = 0x00
TAILLE_DICO_MAX = 255


def apprendre_dictionnaire(donnees: bytes, longueurs=(16, 12, 8), max_entrees=96) -> list[bytes]:
    """Choisit des sous-chaines frequentes, par gain brut (occurrences - 1) * longueur."""
    echantillon = donnees[:16384]
    frequences: Counter = Counter()
    for longueur in longueurs:
        for i in range(len(echantillon) - longueur + 1):
            frequences[echantillon[i:i + longueur]] += 1

    retenus: list[bytes] = []
    occupes: set[bytes] = set()
    for motif, compte in frequences.most_common(4000):
        if len(retenus) >= max_entrees:
            break
        # Gain : on remplace `compte` occurrences par 2 octets chacune.
        if compte < 2 or (compte - 1) * (len(motif) - 2) < len(motif) + 4:
            continue
        # Pas de motif inclus dans un motif deja retenu : il ne serait jamais atteint.
        if any(motif in deja or deja in motif for deja in occupes):
            continue
        retenus.append(motif)
        occupes.add(motif)
    return retenus[:TAILLE_DICO_MAX]


def encoder_dictionnaire(donnees: bytes, dico: list[bytes]) -> bytes:
    index = {motif: i for i, motif in enumerate(dico)}
    sortie = bytearray()
    i = 0
    while i < len(donnees):
        meilleur = None
        for motif in dico:
            if donnees.startswith(motif, i) and (meilleur is None or len(motif) > len(meilleur)):
                meilleur = motif
        if meilleur is not None:
            sortie.append(MARQUEUR)
            sortie.append(index[meilleur] + 1)
            i += len(meilleur)
        else:
            octet = donnees[i]
            sortie.append(octet)
            if octet == MARQUEUR:
                sortie.append(MARQUEUR)
            i += 1
    return bytes(sortie)


def decoder_dictionnaire(donnees: bytes, dico: list[bytes]) -> bytes:
    sortie = bytearray()
    i = 0
    while i < len(donnees):
        octet = donnees[i]
        if octet != MARQUEUR:
            sortie.append(octet)
            i += 1
            continue
        suivant = donnees[i + 1]
        if suivant == 0:
            sortie.append(0)
        else:
            sortie += dico[suivant - 1]
        i += 2
    return bytes(sortie)


# Le dictionnaire est appris UNE fois sur le corpus, puis transmis une seule fois.
DICO = apprendre_dictionnaire(b"".join(p.read_bytes()[:4096] for p in FICHIERS))
EN_TETE_DICO = zlib.compress(
    json.dumps([m.decode("latin-1") for m in DICO]).encode("utf-8"), 9
)
DICO_DECODE = [m.encode("latin-1") for m in json.loads(zlib.decompress(EN_TETE_DICO))]


def _compresser_dico(donnees: bytes) -> bytes:
    return len(EN_TETE_DICO).to_bytes(4, "big") + EN_TETE_DICO + zlib.compress(
        encoder_dictionnaire(donnees, DICO_DECODE), 9
    )


def _decompresser_dico(donnees: bytes) -> bytes:
    taille = int.from_bytes(donnees[:4], "big")
    dico = [m.encode("latin-1") for m in json.loads(zlib.decompress(donnees[4:4 + taille]))]
    return decoder_dictionnaire(zlib.decompress(donnees[4 + taille:]), dico)


CODECS["dict"] = (_compresser_dico, _decompresser_dico)

for temoin in (b"", temoin_texte[:600], temoin_binaire):
    assert _decompresser_dico(_compresser_dico(temoin)) == temoin, "codec dict non inversible"

print(f"dictionnaire appris : {len(DICO)} entrees, {len(EN_TETE_DICO)} o une fois compresse")
print(f"exemples            : {[m.decode('latin-1')[:28] for m in DICO[:6]]}")
print(f"codecs proposes au modele : {sorted(CODECS)}")

dictionnaire appris : 96 entrees, 403 o une fois compresse
exemples            : ['ligne de mesure ', 'igne de mesure r', 'gne de mesure re', 'ne de mesure rep', 'e de mesure repe', ' de mesure repet']
codecs proposes au modele : ['bz2', 'dict', 'lzma', 'store', 'zlib']


### Lecture du résultat

Le dictionnaire a retenu des motifs du corpus — des fragments de lignes CSV, de niveaux de journal, de clés JSON. Il est appris **une fois** et transmis **une fois** : son en-tête compressé est payé au total une seule fois, pas à chaque segment. C'est exactement la structure de coût d'un dictionnaire de `zstd`, et c'est ce qui rend le quatrième codec honnêtement comparable aux trois autres.

Le gain n'est pas garanti sur un segment isolé : un segment de 40 Ko d'octets aléatoires ne contient aucun motif du dictionnaire, donc le codec y ajoute un `0x00` échappé de temps en temps sans rien retirer. C'est au modèle de ne pas le choisir là — et au carnet de publier le résultat même si le modèle se trompe.

### Exercice 2 — un cinquième codec, inversible par construction

Le codec le plus simple qui existe est le **RLE** (*run-length encoding*) : une suite de `n` fois le même octet devient le couple `(octet, n)`. Il est excellent sur les images de plats de couleur unie, catastrophique sur du texte, et surtout il est **trivialement inversible** — ce qui en fait un bon terrain d'exercice.

**Objectif** : compléter `compresser_rle` et `decompresser_rle` de sorte que `decompresser_rle(compresser_rle(x)) == x` pour tout `x`.

**Indice** — un encodage sans ambiguïté : émettre `octet` puis `nombre` sur **deux octets** à chaque fois (donc un maximum de 255 répétitions par groupe). Le décodeur lit la sortie par paires, sans jamais devoir deviner.

**Étape 1** : parcourir l'entrée en comptant les répétitions consécutives. **Étape 2** : à chaque changement, émettre le couple. **Étape 3** : ne pas oublier la dernière série.

**Attention** : ce codec ne doit pas être ajouté à `CODECS` avant d'être correct — sinon le modèle pourrait le choisir pendant le triage et le carnet échouerait au milieu de la passe. La cellule de vérification ne s'exécute que si l'exercice rend quelque chose.

In [8]:
def compresser_rle(donnees: bytes):
    """Exercice 2 : encoder les repetitions consecutives en couples (octet, nombre)."""
    # TODO etudiant : suivre les trois etapes de l'enonce.
    return None


def decompresser_rle(donnees: bytes):
    """Exercice 2 : reconstruire les octets d'origine."""
    # TODO etudiant : lire l'entree par paires.
    return None


_essai = compresser_rle(b"aaabbbbbcc")
if _essai is None:
    print("Exercice 2 a completer : codec RLE")
else:
    for _temoin in (b"", b"a", b"aaa", b"ababab", temoin_binaire):
        assert decompresser_rle(compresser_rle(_temoin)) == _temoin, "RLE non inversible"
    CODECS["rle"] = (compresser_rle, decompresser_rle)
    print(f"RLE injecte et verifie ; codecs : {sorted(CODECS)}")

Exercice 2 a completer : codec RLE


## La recette, et l'invariant de couverture

Une opération est **inversible** et **déclarée**. Le carnet en connaît deux familles :

| Famille | Opération | Effet | Inverse |
|---|---|---|---|
| primitive | `compresser(segment, codec)` | segment → segment compressé en binaire | le codec correspondant |
| intermédiaire | `decouper(segment, mode)` | segment macro → plusieurs segments qualifiés | le recollage propre au mode |

La **recette** est la liste ordonnée des décisions. Elle ne contient aucune donnée du corpus : uniquement des identifiants de segments, des noms de codecs, des noms de modes et, pour les découpages non contigus — colonnes d'un CSV, clés d'un JSON — la colle minimale qui permet de recoller.

**L'invariant de couverture est le cœur du contrat** : un segment original est *couvert* s'il a reçu un traitement final, ou s'il a été découpé en enfants tous couverts. Le calcul est récursif, et il **lève** une exception en nommant les segments oubliés. Une passe qui laisse un segment sans traitement n'est pas une passe incomplète à signaler : c'est une passe échouée.

In [9]:
class CouvertureIncomplete(RuntimeError):
    """Leve quand un segment original n'a recu aucun traitement final."""


def recoller(mode: str, enfants: list[bytes], colle: dict) -> bytes:
    """L'inverse d'un decoupage : reconstruit les octets du segment parent."""
    if mode in ("lignes", "morceaux", "blocs_markdown"):
        return b"".join(enfants)
    if mode == "colonnes_csv":
        colonnes = [e.decode("utf-8").split("\n") for e in enfants]
        n = len(colonnes[0])
        lignes = [colle["sep"].join(col[r] for col in colonnes) for r in range(n)]
        return (colle["nl"].join(lignes) + (colle["nl"] if colle["final_nl"] else "")).encode("utf-8")
    if mode == "cles_json":
        objet = {cle: json.loads(e.decode("utf-8")) for cle, e in zip(colle["cles"], enfants)}
        return (json.dumps(objet, indent=2, ensure_ascii=False)
                + ("\n" if colle["final_nl"] else "")).encode("utf-8")
    raise ValueError(f"mode de recolle inconnu : {mode}")


def decouper(seg: Segment, mode: str, prefixe: str) -> tuple[list[Segment], dict]:
    """Operation intermediaire : un segment macro -> des segments qualifies + la colle."""
    if mode not in decoupages_possibles(seg):
        raise ValueError(f"{mode} ne s'applique pas a {seg.id}")
    profondeur = seg.profondeur + 1

    if mode in ("lignes", "morceaux", "blocs_markdown"):
        if mode == "lignes":
            # keepends=True : chaque groupe garde ses propres retours a la ligne,
            # donc la concatenation b"".join des groupes est exacte. Un split
            # naif par \n perd les joints aux frontieres de groupes (mesure :
            # une vraie passe a reconstruit app.log ampute de 3 octets).
            lignes = seg.data.splitlines(keepends=True)
            parts = max(1, min(4, len(lignes) // 20))
            taille = max(1, len(lignes) // parts)
            morceaux = [b"".join(lignes[i:i + taille]) for i in range(0, len(lignes), taille)]
        elif mode == "morceaux":
            parts = 2
            taille = max(1, len(seg.data) // parts)
            morceaux = [seg.data[i:i + taille] for i in range(0, len(seg.data), taille)]
        else:
            texte = seg.data.decode("utf-8")
            positions = [m.start() for m in re.finditer(r"^##? ", texte, re.M)]
            bornes = sorted({0, *positions, len(texte)})
            morceaux = [texte[a:b].encode("utf-8") for a, b in zip(bornes, bornes[1:])]
        enfants = [Segment(f"{prefixe}.{i}", m, f"{seg.genre}/{mode}", seg.id, profondeur)
                   for i, m in enumerate(morceaux) if m]
        return enfants, {"mode": mode}

    if mode == "colonnes_csv":
        texte = seg.data.decode("utf-8")
        final_nl = texte.endswith("\n")
        lignes = (texte[:-1] if final_nl else texte).split("\n")
        cellules = [ligne.split(",") for ligne in lignes]
        enfants = []
        for j in range(len(cellules[0])):
            colonne = "\n".join(rangee[j] for rangee in cellules).encode("utf-8")
            enfants.append(Segment(f"{prefixe}.c{j}", colonne,
                                   f"{seg.genre}/colonne:{cellules[0][j]}", seg.id, profondeur))
        return enfants, {"mode": mode, "sep": ",", "nl": "\n", "final_nl": final_nl}

    if mode == "cles_json":
        texte = seg.data.decode("utf-8")
        final_nl = texte.endswith("\n")
        objet = json.loads(texte)
        enfants = [Segment(f"{prefixe}.k{i}",
                           json.dumps(valeur, indent=2, ensure_ascii=False).encode("utf-8"),
                           f"{seg.genre}/cle:{cle}", seg.id, profondeur)
                   for i, (cle, valeur) in enumerate(objet.items())]
        return enfants, {"mode": mode, "cles": list(objet), "final_nl": final_nl}

    raise ValueError(f"mode inconnu : {mode}")


# Contrat d'inversibilite des decoupages, symetrique de celui des codecs :
# decouper puis recoller doit redonner les octets d'origine, pour chaque mode.
# Les temoins couvrent le cas limite qui a fait echouer une vraie passe avant ce
# garde-fou : un texte de plus de 40 lignes dont la derniere se termine par un
# retour a la ligne, decoupe en plusieurs groupes -- c'est au joint entre deux
# groupes qu'un split naif perd ses octets de joint.
_lignes_avec_final = "".join(f"ligne-{i:02d}\n" for i in range(45)).encode("utf-8")
_lignes_sans_final = "\n".join(f"ligne-{i:02d}" for i in range(45)).encode("utf-8")
_paires_decoupage = [
    ("lignes", _lignes_avec_final, "texte/journal.log"),
    ("lignes", _lignes_sans_final, "texte/journal.log"),
    ("morceaux", bytes(range(256)) * 3, "bin/brut.bin"),
    ("blocs_markdown", b"# Titre\n\ntexte\n\n## Sous-titre\n\ntexte\n", "texte/notes.md"),
    ("colonnes_csv", b"alpha,beta\n1,2\n3,4\n", "donnees/ventes.csv"),
    ("cles_json", (json.dumps({"a": 1, "b": [1, 2]}, indent=2, ensure_ascii=False)
                   + "\n").encode("utf-8"), "donnees/clients.json"),
]
for _mode, _donnees, _genre in _paires_decoupage:
    _temoin = Segment(f"temoin/{_genre}", _donnees, _genre)
    assert _mode in decoupages_possibles(_temoin), f"temoin invalide pour {_mode}"
    _enfants, _colle = decouper(_temoin, _mode, "temoin")
    assert _enfants, f"decoupage vide pour {_mode}"
    assert recoller(_mode, [e.data for e in _enfants], _colle) == _donnees, \
        f"decoupage non inversible : {_mode}"
print("decoupages inversibles verifies : lignes (avec et sans retour final), "
      "morceaux, blocs_markdown, colonnes_csv, cles_json")


def _couverture(sid: str, recette: dict, vus: set) -> tuple[bool, list[str]]:
    """Un segment est couvert s'il est compresse, ou decoupe en enfants tous couverts."""
    if sid in vus:
        raise CouvertureIncomplete(f"cycle dans la recette autour de {sid}")
    vus = vus | {sid}
    entree = recette.get(sid)
    if entree is None:
        return False, [sid]
    if entree["op"] == "compresser":
        return True, []
    manquants: list[str] = []
    for enfant in entree["enfants"]:
        ok, absents = _couverture(enfant, recette, vus)
        if not ok:
            manquants.extend(absents)
    return not manquants, manquants


def verifier_couverture(racines: list[str], recette: dict) -> None:
    """Invariant : leve en nommant tout segment original non traite."""
    manquants: list[str] = []
    for sid in racines:
        ok, absents = _couverture(sid, recette, set())
        if not ok:
            manquants.extend(absents)
    if manquants:
        raise CouvertureIncomplete(
            f"{len(manquants)} segment(s) sans traitement final : {manquants[:6]}"
        )


print(f"{len(decoupages_possibles(SEGMENTS['ventes/ventes.csv#0']))} decoupages possibles pour le CSV")
print(f"exemple : {decoupages_possibles(SEGMENTS['data/mesures.bin#0'])} pour le binaire")

decoupages inversibles verifies : lignes (avec et sans retour final), morceaux, blocs_markdown, colonnes_csv, cles_json
3 decoupages possibles pour le CSV
exemple : ['lignes', 'morceaux'] pour le binaire


### Lecture du résultat

Le CSV accepte `colonnes_csv` ; le binaire ne l'accepte pas, et ne propose même pas de découpage markdown ou JSON. Cette asymétrie est calculée à partir du type du segment, pas déclarée à la main : c'est elle qui empêche le modèle de proposer une opération inapplicable.

Le binaire conserve `lignes` et `morceaux` — découper des octets reste possible même sans structure connue, simplement cela n'apporte rien. Le modèle devra le découvrir, ou le carnet le lui montrera par la mesure.

### L'invariant doit échouer quand il le faut

Un invariant qu'on n'a jamais vu échouer n'est pas un invariant, c'est une fonction qui rend `True`. La cellule suivante construit **exprès** une recette incomplète — un CSV découpé en colonnes dont une seule est compressée — et vérifie que `verifier_couverture` lève en **nommant** les colonnes oubliées.

C'est le contrôle du contrôle : sans lui, l'invariant de la passe réelle pourrait passer pour de mauvaises raisons.

In [10]:
temoin_id = "ventes/ventes.csv#0"
temoin_enfants, temoin_colle = decouper(SEGMENTS[temoin_id], "colonnes_csv", "temoin")

recette_incomplete = {
    temoin_id: {"op": "decouper", "mode": "colonnes_csv", "colle": temoin_colle,
                "enfants": [e.id for e in temoin_enfants]},
}
recette_incomplete[temoin_enfants[0].id] = {"op": "compresser", "codec": "zlib"}

try:
    verifier_couverture([temoin_id], recette_incomplete)
    raise AssertionError("l'invariant n'a pas vu les colonnes non traitees")
except CouvertureIncomplete as erreur:
    print(f"invariant declenche comme attendu : {erreur}")

# Controle positif : la meme recette, toutes colonnes traitees, doit passer.
for enfant in temoin_enfants:
    recette_incomplete[enfant.id] = {"op": "compresser", "codec": "zlib"}
verifier_couverture([temoin_id], recette_incomplete)
print("controle positif : recette complete acceptee")

invariant declenche comme attendu : 5 segment(s) sans traitement final : ['temoin.c1', 'temoin.c2', 'temoin.c3', 'temoin.c4', 'temoin.c5']
controle positif : recette complete acceptee


### Lecture du résultat

L'invariant a nommé les colonnes oubliées, puis a accepté la même recette une fois toutes les colonnes traitées. Le contrôle positif compte autant que le négatif : un invariant qui refuserait **tout** serait aussi inutile qu'un invariant qui accepte tout, et le carnet ne saurait pas distinguer les deux sans l'avoir éprouvé dans les deux sens.

## Le harnais Semantic Kernel

Le carnet 07 écrivait ses schémas d'outils à la main en JSON et postait ses requêtes avec `urllib`. Le dépôt documente déjà le harnais qui fait cela proprement — `Kernel`, plugins `@kernel_function`, `FunctionChoiceBehavior.Auto()` — dans la série GenAI (voir `GenAI/Texte/18_Semantic_Kernel_Plugins.ipynb` et le cas d'étude Medical-Chatbot). Ce carnet **réutilise** ce harnais plutôt que d'en écrire un second.

Le plugin expose deux fonctions, exactement les deux familles d'opérations. Le modèle les voit comme des outils ; quand il en choisit une, **la fonction Python s'exécute en processus** et son résultat lui est renvoyé. Ce n'est pas une simulation : le compteur d'appels et la trace des décisions plus bas sont la preuve que la fonction a réellement tourné.

Un piège mesuré sur cette pile : le modèle servi est un modèle de **raisonnement**, et il consomme des jetons de raisonnement avant de choisir. Avec `max_tokens` trop bas, il ne rend ni texte ni appel d'outil — la réponse est vide. La cellule fixe donc un budget large ; c'est un fait mesuré, pas une marge de confort.

Un second comportement, mesuré lors de la première exécution : sur le segment d'octets pseudo-aléatoires, le modèle a répondu en prose — jugeant, à raison, que rien n'y est compressible — **sans appeler la fonction**. `demander_decision` traite ce cas par une **relance unique** : elle cite la réponse en texte libre au modèle et redemande l'appel d'outil. La relance est un appel réel, compté dans `COMPTEUR` ; si le modèle persiste, la passe devient rouge — elle ne choisit pas à sa place.

In [11]:
class PluginSegments:
    """Plugin Semantic Kernel : les deux operations inversibles, exposees au modele.

    Le plugin ne connait que le segment courant. Il valide l'operation demandee
    et, si elle ne s'applique pas, renvoie la raison au modele plutot que de
    lever : le modele peut alors se corriger dans le meme tour.
    """

    def __init__(self, etat: dict):
        self.etat = etat

    @kernel_function(
        name="compresser",
        description=(
            "Applique un traitement FINAL a un segment : il devient un binaire compresse "
            "et ne sera plus jamais decoupe. Codecs disponibles : zlib, bz2, lzma, dict, store."
        ),
    )
    def compresser(self, codec: str) -> str:
        seg = self.etat["segment"]
        if codec not in CODECS:
            return f"codec inconnu {codec!r} ; disponibles : {sorted(CODECS)}"
        self.etat["decision"] = {"op": "compresser", "codec": codec}
        return f"accepte : {seg.id} compresse avec {codec}"

    @kernel_function(
        name="decouper",
        description=(
            "Separe un segment macro en sous-segments qualifies, qui recevront chacun "
            "un traitement ulterieur. Utiliser les modes listes dans `decoupages_possibles`."
        ),
    )
    def decouper(self, mode: str) -> str:
        seg = self.etat["segment"]
        possibles = decoupages_possibles(seg)
        if mode not in possibles:
            return f"mode {mode!r} inapplicable a {seg.id} ; possibles : {possibles}"
        self.etat["decision"] = {"op": "decouper", "mode": mode}
        return f"accepte : {seg.id} decoupe en {mode}"


async def demander_decision(seg: Segment, carte: str) -> dict | None:
    """Un aller-retour avec le modele pour un segment. Compte l'appel reel."""
    global DERNIER_TEXTE_LIBRE
    etat = {"segment": seg, "decision": None}
    kernel = kernel_pour_decision(etat)

    reglages = OpenAIPromptExecutionSettings(
        max_tokens=3000,
        temperature=0.0,
        function_choice_behavior=FunctionChoiceBehavior.Auto(
            auto_invoke=True, maximum_auto_invoke_attempts=2
        ),
    )
    historique = ChatHistory()
    historique.add_system_message(
        "Tu es un moteur de triage de compression. Pour le segment courant, tu appelles "
        "EXACTEMENT UNE des deux fonctions du plugin `segments` : soit `compresser` (traitement "
        "final, irreversible), soit `decouper` (separation en sous-segments, chacun traite "
        "ensuite). Tu ne reponds jamais en texte libre : tu appelles une fonction, puis tu "
        "conclus en une phrase."
    )
    historique.add_user_message(
        "Carte du corpus (segments deja traites omis) :\n"
        f"{carte}\n\n"
        f"Segment a traiter maintenant : {json.dumps(vue(seg), ensure_ascii=False)}\n\n"
        f"Codecs : {sorted(CODECS)}. Modes applicables : {decoupages_possibles(seg)}.\n"
        "Choisis : `decouper` si la structure interne du segment permet d'isoler des parties "
        "de nature differente (colonnes, cles, blocs) ; sinon `compresser` avec le codec le "
        "plus adapte a ce contenu. Un segment binaire sans structure doit etre compresse, "
        "pas decoupe."
    )

    service = kernel.get_service(SERVICE)
    COMPTEUR["appels"] += 1
    reponse = await service.get_chat_message_contents(
        chat_history=historique, settings=reglages, kernel=kernel
    )

    if etat["decision"] is None:
        # Le modele a repondu en texte libre, ou n'a rien produit. Une relance
        # unique cite sa reponse et redemande l'appel d'outil : c'est un vrai
        # tour d'agent, compte comme tel dans COMPTEUR.
        texte_libre = str(reponse[0]).strip()[:200] or "(reponse vide)"
        DERNIER_TEXTE_LIBRE = texte_libre
        historique.add_assistant_message(texte_libre)
        historique.add_user_message(
            "Tu viens de repondre en texte libre au lieu d'appeler une fonction. "
            "Appelle maintenant exactement une fonction du plugin : "
            f"compresser(codec=...) ou decouper(mode=...). "
            f"Modes applicables ici : {decoupages_possibles(seg)}."
        )
        reglages.max_tokens = 4000
        COMPTEUR["appels"] += 1
        await service.get_chat_message_contents(
            chat_history=historique, settings=reglages, kernel=kernel
        )
    return etat["decision"]


aoai = AsyncOpenAI(api_key=CLE, base_url=BASE)
DERNIER_TEXTE_LIBRE = ""


def kernel_pour_decision(etat: dict) -> Kernel:
    """Un kernel NEUF par decision, portant exactement un plugin.

    Mesure a l'appui : avec un kernel partage, chaque appel y ajoute une paire
    compresser/decouper de plus ; des le deuxieme segment, le modele voit
    quatre outils dont une paire liee a l'etat du segment PRECEDENT, et il
    refuse d'en appeler aucun (deux executions reelles, meme refus au meme
    segment). Reconstruire le kernel par decision elimine la classe entiere.
    """
    k = Kernel()
    k.add_service(OpenAIChatCompletion(ai_model_id=MODELE, async_client=aoai, service_id=SERVICE))
    k.add_plugin(PluginSegments(etat), plugin_name="segments")
    return k


print(f"kernel pret : service {SERVICE!r} -> {MODELE} (un kernel neuf par decision)")

kernel pret : service 'vllm-local' -> qwen3.6-35b-a3b (un kernel neuf par decision)


### Lecture du résultat

Le kernel est construit sur le client OpenAI asynchrone pointé sur l'endpoint local — c'est la forme déjà employée par la série GenAI, à ceci près que l'endpoint est ici la pile vLLM de la maison plutôt qu'un fournisseur externe.

Deux détails du harnais méritent d'être lus. D'abord, **un kernel neuf par décision** : les plugins s'ajoutent à un kernel et ne s'en retirent pas, et nous l'avons mesuré — avec un kernel partagé, au deuxième segment le modèle voyait quatre fonctions, dont une paire `compresser`/`decouper` encore liée à l'état du segment précédent, et il refusait d'en appeler aucune (deux exécutions réelles, même refus au même segment). Reconstruire le kernel à chaque décision élimine la classe d'erreur entière. Ensuite, une opération inapplicable **ne lève pas** : elle rend un message que le modèle lit dans le même tour. C'est ce qui permet à `maximum_auto_invoke_attempts=2` d'avoir un sens — la seconde tentative du modèle dispose de la raison de l'échec de la première.

## Le triage : une boucle jusqu'à couverture complète

Chaque tour traite **un** segment en attente : le modèle reçoit la carte du corpus, la vue du segment, et choisit. Le carnet applique la décision, met à jour la file, et recommence jusqu'à ce qu'il ne reste plus rien.

Deux bornes protègent la boucle, et elles ont des rôles différents :

- **la profondeur** (`PROFONDEUR_MAX = 2`) est structurelle : elle empêche un découpage infini ;
- **le nombre d'appels** (`BORNE_APPELS`) est un garde-fou de coût. Si la boucle l'atteint alors qu'il reste des segments, la passe **échoue** — ce n'est pas une fin normale, c'est un triage qui n'a pas abouti.

La couverture est vérifiée **après** la boucle, par l'invariant, et non supposée parce que la file est vide. Une file vide et une couverture complète sont deux choses différentes : la seconde se prouve.

In [12]:
BORNE_APPELS = 64


async def trier(segments: list[Segment]) -> dict:
    """Boucle de triage : le modele est rappele tant qu'un segment n'a pas de traitement final."""
    recette: dict = {}
    a_traiter = list(segments)
    tous: dict[str, Segment] = {s.id: s for s in segments}

    while a_traiter:
        if COMPTEUR["appels"] >= BORNE_APPELS:
            raise RuntimeError(
                f"triage interrompu a {COMPTEUR['appels']} appels avec "
                f"{len(a_traiter)} segment(s) non traites — la passe echoue."
            )
        seg = a_traiter.pop(0)
        carte = json.dumps(
            [{"id": s.id, "genre": s.genre, "octets": len(s.data),
              "structure": structure_interne(s)} for s in a_traiter[:24]],
            ensure_ascii=False,
        )
        decision = await demander_decision(seg, carte)
        if decision is None:
            detail = f" ; dernier texte libre du modele : {DERNIER_TEXTE_LIBRE[:120]!r}" if DERNIER_TEXTE_LIBRE else ""
            raise RuntimeError(f"le modele n'a choisi aucune operation pour {seg.id} — passe rouge{detail}")

        if decision["op"] == "compresser":
            compresse, _ = CODECS[decision["codec"]]
            seg.data = compresse(seg.data)
            seg.genre = f"{seg.genre}/compresse:{decision['codec']}"
            recette[seg.id] = {"op": "compresser", "codec": decision["codec"]}
        else:
            enfants, colle = decouper(seg, decision["mode"], seg.id.rstrip("#0"))
            recette[seg.id] = {"op": "decouper", "mode": decision["mode"],
                               "colle": colle, "enfants": [e.id for e in enfants]}
            for enfant in enfants:
                tous[enfant.id] = enfant
            a_traiter.extend(enfants)
        # Une ligne par decision, affichee AU FUR ET A MESURE : chaque appel au
        # modele est un aller-retour de plusieurs secondes, et la passe complete
        # en compte une vingtaine -- sans ce flush, la cellule resterait muette
        # (et le fichier de sortie avec) pendant toute la passe.
        print(f"[appel {COMPTEUR['appels']:3d}] {seg.id:32s} -> "
              f"{decision['op']:10s} {decision.get('codec') or decision.get('mode')}",
              flush=True)

    verifier_couverture([s.id for s in segments], recette)
    return {"recette": recette, "segments": tous}


resultat = await trier(list(SEGMENTS.values()))
recette = resultat["recette"]
print(f"appels LLM reellement effectues : {COMPTEUR['appels']}")
print(f"segments dans l'arbre final     : {len(resultat['segments'])}")
print()
for sid, entree in recette.items():
    if entree["op"] == "compresser":
        print(f"  {sid:44s} compresser  {entree['codec']}")
    else:
        print(f"  {sid:44s} decouper    {entree['mode']} -> {len(entree['enfants'])} segments")

[appel   1] config/settings.json#0           -> compresser zlib


[appel   2] data/mesures.bin#0               -> compresser store


[appel   3] data/mixte.dat#0                 -> compresser lzma


[appel   4] docs/notes.md#0                  -> compresser lzma


[appel   5] logs/app.log#0                   -> decouper   lignes


[appel   6] src/analyse.py#0                 -> compresser zlib


... (11 outputs left out)

[appel  18] ventes/ventes.csv.c3             -> compresser lzma


[appel  19] ventes/ventes.csv.c4             -> compresser lzma


[appel  20] ventes/ventes.csv.c5             -> compresser dict


appels LLM reellement effectues : 20
segments dans l'arbre final     : 20

  config/settings.json#0                       compresser  zlib
  data/mesures.bin#0                           compresser  store
  data/mixte.dat#0                             compresser  lzma
  docs/notes.md#0                              compresser  lzma
  logs/app.log#0                               decouper    lignes -> 4 segments
  src/analyse.py#0                             compresser  zlib
  ventes/clients.json#0                        decouper    cles_json -> 2 segments
  ventes/ventes.csv#0                          decouper    colonnes_csv -> 6 segments
  logs/app.log.0                               compresser  lzma
  logs/app.log.1                               compresser  lzma
  logs/app.log.2                               compresser  lzma
  logs/app.log.3                               compresser  lzma
  ventes/clients.json.k0                       compresser  zlib
  ventes/clients.json.k1           

### Lecture du résultat

Le nombre d'appels affiché est celui des appels **réellement partis** vers le modèle : le compteur est incrémenté juste avant l'envoi, dans `demander_decision`, et nulle part ailleurs. C'est une mesure, pas une estimation.

La recette se lit comme un arbre : les lignes `compresser` sont les feuilles — traitement final, irréversible — et les lignes `decouper` sont les nœuds internes. La couverture vérifiée à la fin de `trier` porte sur les racines, et elle est récursive : un nœud interne n'est couvert que si tous ses enfants le sont.

Les choix du modèle méritent d'être confrontés à ce que la structure annonçait. Sur un CSV dont la carte donne les six colonnes, un découpage `colonnes_csv` sépare une colonne de catégories d'une colonne de montants ; sur un binaire sans structure reconnue, la seule décision raisonnable est `compresser` — et `store` sur les octets aléatoires, puisque rien n'y est compressible. Le modèle ne le fait pas toujours, et le carnet ne corrige pas ses décisions : les chiffres de la comparaison sont les siens.

## Décompresser — sans aucun appel au modèle

C'est ici que la thèse du carnet se vérifie. La décompression **rejoue la recette à l'envers** : elle n'a besoin ni du modèle, ni de la carte du corpus, ni d'une heuristique. Chaque décision porte son inverse, et le recollage est déterministe.

Le compteur d'appels est remis à zéro **juste avant** la reconstruction, puis relu **juste après**. Un compteur à zéro n'est une preuve que si l'on sait qu'il aurait pu bouger : comme la même fonction de comptage est celle qui a produit le nombre d'appels du triage quelques cellules plus haut, un appel qui se glisserait ici serait vu.

La preuve finale n'est pas « ça décompresse » : c'est l'**égalité binaire**, fichier par fichier, avec les octets d'origine — comparés par empreinte pour être lisibles.

In [13]:
import hashlib


def reconstruire(sid: str, recette: dict, segments: dict) -> bytes:
    """Rejoue la recette a l'envers. Aucun appel au modele, aucune heuristique."""
    entree = recette.get(sid)
    if entree is None:
        raise CouvertureIncomplete(f"{sid} absent de la recette")
    if entree["op"] == "compresser":
        _, decompresser = CODECS[entree["codec"]]
        return decompresser(segments[sid].data)
    enfants = [reconstruire(e, recette, segments) for e in entree["enfants"]]
    return recoller(entree["mode"], enfants, entree["colle"])


COMPTEUR["appels_tri"] = COMPTEUR["appels"]  # le total du triage survit au reset
COMPTEUR["appels"] = 0          # remise a zero : le compteur devient l'oracle
reconstruits = {sid: reconstruire(sid, recette, resultat["segments"]) for sid in SEGMENTS}
appels_pendant_decompression = COMPTEUR["appels"]

egalites = []
for chemin in FICHIERS:
    relatif = chemin.relative_to(RACINE).as_posix()
    origine = chemin.read_bytes()
    obtenu = reconstruits[f"{relatif}#0"]
    egalites.append((relatif, origine == obtenu, hashlib.sha256(obtenu).hexdigest()[:16]))

for relatif, egal, empreinte in egalites:
    print(f"  {'IDENTIQUE' if egal else 'DIFFERENT':10s} {relatif:26s} sha256[:16]={empreinte}")

assert appels_pendant_decompression == 0, "la decompression a appele le modele"
assert all(egal for _, egal, _ in egalites), "au moins un fichier differe de son original"
print()
print(f"appels LLM pendant la decompression : {appels_pendant_decompression}")
print(f"{len(egalites)}/{len(egalites)} fichiers reconstruits a l'identique, "
      f"total {sum(len(v) for v in reconstruits.values())} o")

  IDENTIQUE  config/settings.json       sha256[:16]=b2b460fa50591305
  IDENTIQUE  data/mesures.bin           sha256[:16]=8ac1d0c4b56ecc39
  IDENTIQUE  data/mixte.dat             sha256[:16]=4c2587020e2dcfab
  IDENTIQUE  docs/notes.md              sha256[:16]=8be478aa358bbcc7
  IDENTIQUE  logs/app.log               sha256[:16]=9c82e5f6d750df2a
  IDENTIQUE  src/analyse.py             sha256[:16]=009b8b956d5e7700
  IDENTIQUE  ventes/clients.json        sha256[:16]=2dad6acb5cf0869d
  IDENTIQUE  ventes/ventes.csv          sha256[:16]=750e8428f0cbe10a

appels LLM pendant la decompression : 0
8/8 fichiers reconstruits a l'identique, total 110688 o


### Lecture du résultat

Tous les fichiers sont identiques à leurs originaux, **et** le compteur affiche zéro. Les deux ensemble sont la preuve : l'égalité binaire seule ne dirait pas si le modèle a été consulté, et le compteur seul ne dirait pas si la reconstruction est fidèle.

L'empreinte tronquée à seize caractères rend la comparaison lisible sans noyer la sortie ; la comparaison réelle, elle, porte sur les octets complets (`origine == obtenu`).

Notez ce que la recette ne contient **pas** : aucune donnée du corpus. Un décodeur qui posséderait le corpus d'origine pourrait toujours « reconstruire », et ne prouverait rien. Ici le décodeur n'a que les artefacts compressés, les identifiants et les colles de découpage.

## La thèse à réfuter : le triage bat-il le traitement indifférencié ?

Toute la construction qui précède ne vaut que si elle répond à une question mesurable. La voici, posée par le Concern à l'origine de ce carnet : **consacrer le modèle au triage et aux transformations de structure permet-il de mieux compresser qu'un traitement indifférencié ?**

Trois totaux sont mis en regard, tous en octets, tous incluant ce qu'un comparaison honnête doit inclure :

1. **triage** — somme des artefacts de toutes les feuilles, **plus** l'en-tête du dictionnaire, **plus** la recette sérialisée et compressée. Une recette gratuite n'existe pas : elle fait partie du fichier qu'il faut transmettre pour pouvoir décompresser.
2. **bloc indifférencié, même codec** — les fichiers mis bout à bout, compressés d'un seul tenant avec le codec que le triage a le plus employé. C'est le traitement naïf équitable.
3. **bloc indifférencié, meilleur codec choisi après coup** — on essaie les quatre codecs sur le tout et on garde le meilleur. C'est une **borne favorable au traitement naïf** : personne ne fait cela sans essayer tous les codecs sur tout le contenu, mais elle donne au camp d'en face tout le bénéfice du doute.

Le verdict est publié **même s'il est défavorable au triage**. Un carnet qui ne publierait que les cas où il gagne ne mesurerait rien.

In [14]:
import statistics


def taille_recette(recette: dict) -> int:
    """La recette doit voyager : son cout est compte, compresse comme elle le serait."""
    brut = json.dumps(recette, ensure_ascii=False, sort_keys=True).encode("utf-8")
    return len(zlib.compress(brut, 9))


def tout_en_un() -> bytes:
    return b"".join(p.read_bytes() for p in FICHIERS)


feuilles = [s for s in resultat["segments"].values()
            if recette[s.id]["op"] == "compresser"]
artefacts = sum(len(s.data) for s in feuilles)
cout_recette = taille_recette(recette)
cout_dico = len(EN_TETE_DICO)
total_triage = artefacts + cout_recette + cout_dico

# Codec majoritaire du triage : c'est le codec que le camp naif est en droit d'utiliser.
codecs_choisis = Counter(e["codec"] for e in recette.values() if e["op"] == "compresser")
codec_maj = codecs_choisis.most_common(1)[0][0]
bloc = tout_en_un()

total_naif_meme = len(CODECS[codec_maj][0](bloc))
totaux_par_codec = {nom: len(CODECS[nom][0](bloc)) for nom in CODECS}
meilleur = min(totaux_par_codec, key=totaux_par_codec.get)
total_naif_meilleur = totaux_par_codec[meilleur]

print(f"corpus complet                      : {len(bloc):8d} o")
print()
print(f"  triage LLM                        : {total_triage:8d} o "
      f"({artefacts} artefacts + {cout_recette} recette + {cout_dico} dictionnaire)")
print(f"  bloc indifferentie, {codec_maj:5s}        : {total_naif_meme:8d} o")
print(f"  bloc indifferentie, meilleur ({meilleur:4s}) : {total_naif_meilleur:8d} o")
print()
print(f"codecs choisis par le triage : {dict(codecs_choisis)}")
print(f"tailles par codec sur le bloc : {totaux_par_codec}")
print()
gain = total_naif_meme - total_triage
print(f"triage vs bloc/meme codec     : {gain:+8d} o  ({gain / total_naif_meme:+.1%})")
gain2 = total_naif_meilleur - total_triage
print(f"triage vs bloc/meilleur codec : {gain2:+8d} o  ({gain2 / total_naif_meilleur:+.1%})")
verdict = "LE TRIAGE GAGNE" if gain2 > 0 else "LE TRIAGE PERD"
print()
print(f"VERDICT : {verdict} face a la borne la plus favorable au traitement naif")

corpus complet                      :   110688 o

  triage LLM                        :    67299 o (66540 artefacts + 356 recette + 403 dictionnaire)
  bloc indifferentie, lzma         :    64888 o
  bloc indifferentie, meilleur (lzma) :    64888 o

codecs choisis par le triage : {'zlib': 3, 'store': 1, 'lzma': 10, 'dict': 3}
tailles par codec sur le bloc : {'zlib': 66341, 'bz2': 66724, 'lzma': 64888, 'store': 110688, 'dict': 66561}

triage vs bloc/meme codec     :    -2411 o  (-3.7%)
triage vs bloc/meilleur codec :    -2411 o  (-3.7%)

VERDICT : LE TRIAGE PERD face a la borne la plus favorable au traitement naif


### Lecture du résultat — le verdict, quel qu'il soit

Trois totaux, un verdict, et une conclusion qui n'est pas négociable avec le résultat.

**Ce qu'il faut regarder avant le verdict.** Le total du triage paie trois choses que le traitement naïf ne paie pas : la recette (une décision par segment, sérialisée et compressée), l'en-tête du dictionnaire, et le coût de découpage lui-même — une colonne de CSV extraite n'est plus contiguë, donc le recolle coûte de la recette. Ces coûts sont **dans** le total, pas à côté : c'est ce qui rend la comparaison honnête, et c'est aussi ce qui explique qu'un triage intelligent puisse perdre.

**Ce qu'un gain ou une perte signifie vraiment.** Un gain établi sur ce corpus dirait que la structure de ces fichiers — colonnes, clés, lignes de journal — est une redondance que les codecs génériques ne voient pas d'un bloc. Une perte dirait le contraire : que les trois codecs de la bibliothèque standard exploitent déjà ces régularités sur le bloc entier, et que le prix du triage n'est pas récupéré. Les deux verdicts sont des résultats ; seul un verdict fabriqué ne le serait pas.

### Exercice 3 — la borne qui manque : meilleur codec par fichier

La comparaison ci-dessus offre au traitement naïf **un seul codec pour tout le corpus**. Ce n'est pas la borne la plus généreuse qu'on puisse lui construire : un adversaire sérieux choisirait, pour chaque fichier, le codec qui le compresse le mieux — sans structure, sans modèle, mais avec un essai systématique.

**Objectif** : compléter `meilleur_par_fichier` pour qu'elle rende la taille totale obtenue en compressant **chaque fichier séparément** avec son meilleur codec, plus la taille de la table qui dit quel codec va avec quel fichier.

**Indice** — les fichiers sont dans `FICHIERS` ; leurs noms relatifs s'obtiennent par `chemin.relative_to(RACINE).as_posix()`. La table des choix se sérialise et se compresse exactement comme la recette, avec `taille_recette`.

**Étape 1** : pour chaque fichier, essayer tous les `CODECS` et retenir le plus petit. **Étape 2** : sommer ces minima. **Étape 3** : ajouter le coût de la table `{nom: codec}`.

**Question à trancher vous-même** : si le triage perd encore face à cette borne-là, qu'est-ce que cela dit de l'hypothèse du Concern — et qu'est-ce que cela ne dit pas ? (Un corpus de huit fichiers n'est pas un corpus de mille.)

In [15]:
def meilleur_par_fichier():
    """Exercice 3 : borne « meilleur codec par fichier », plus couteuse a etablir."""
    # TODO etudiant : suivre les trois etapes de l'enonce.
    return None


_borne = meilleur_par_fichier()
if _borne is None:
    print("Exercice 3 a completer : borne meilleur-codec-par-fichier")
else:
    print(f"bloc indifferentie, meilleur codec par fichier : {_borne} o")
    print(f"triage : {total_triage} o -> ecart {_borne - total_triage:+d} o")

Exercice 3 a completer : borne meilleur-codec-par-fichier


## Ce que ce carnet a établi

| Acquis | Où il se vérifie |
|---|---|
| Le modèle choisit et **exécute** réellement des fonctions, sous Semantic Kernel | le compteur d'appels du triage, et la trace des décisions |
| Les opérations sont **inversibles**, et chaque inverse est exercé | la vérification de contrat sur les quatre codecs, puis l'égalité binaire |
| La structure des formats est **présentée** au modèle, pas devinée | `structure_interne` : colonnes, clés, niveaux, titres |
| La couverture est un **invariant**, pas une espérance | le contrôle négatif qui la fait échouer, puis la vérification en fin de passe |
| La décompression **n'appelle personne** | le compteur remis à zéro, puis relu à zéro |
| La comparaison est **publiable dans les deux sens** | les trois totaux, recette et dictionnaire compris |

### Ce que ce carnet ne prétend pas

Le corpus fait huit fichiers. Un gain ou une perte sur huit fichiers ne s'extrapole pas à un corpus réel : les codecs de la bibliothèque standard ont des fenêtres de plusieurs mégaoctets, et un corpus plus gros leur donne un avantage que ce banc ne leur donne pas encore. Le résultat publié vaut pour **ce** corpus, mesuré ici, et la cellule de comparaison donne de quoi le réfuter en changeant le corpus.

Le triage lui-même n'est pas évalué à sa juste valeur par ce banc : le carnet mesure ce que le modèle a choisi, il ne mesure pas ce qu'un triage parfait aurait rapporté. Un modèle qui découperait chaque CSV par colonnes et compresserait chaque segment avec son meilleur codec ferait mieux que ce qui est mesuré ici — et c'est précisément l'écart entre les deux qu'un prochain carnet pourrait quantifier.

### Pour continuer

- changer le corpus (plus de fichiers, formats plus variés) et regarder le verdict se déplacer ;
- comparer deux modèles servis par le même endpoint sur le même corpus : mêmes décisions ou non ;
- instrumenter le coût en jetons du triage, qui n'est pas compté ici et qui déciderait, en pratique, de l'intérêt du procédé.

Le carnet [07](Compression-07-Triage-LLM-Operations-Reelles-Python.ipynb) reste la première expérience, avec ses schémas d'outils écrits à la main ; celui-ci en est l'approfondissement sous harnais, avec la couverture prouvée.

In [16]:
# Le repertoire de travail est propre a cette execution : on le retire.
shutil.rmtree(RACINE, ignore_errors=True)
print("repertoire de travail retire")
print(f"recapitulatif : {COMPTEUR['appels_tri']} appels LLM au total (tri), 0 pendant la decompression")

repertoire de travail retire
recapitulatif : 20 appels LLM au total (tri), 0 pendant la decompression
